# Вариационный автокодировщик FashionMNIST — версия 2.0

    **Цель:** Разобрать вероятностное скрытое пространство и reparameterization trick.

    **Алгоритм:** VAE: encoder → μ/logσ² → случайный z → decoder; loss = BCE + KL. В исходном коде использовался FashionMNIST, а не цифры MNIST.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('vae-mnist-pytorch')


## Данные и устройство


In [ ]:
from ml_lab.runtime import torch_setup, demo_images
torch, device = torch_setup(cfg)
from torchvision import datasets, transforms
from torchvision.utils import save_image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import matplotlib.pyplot as plt
if cfg.source == 'demo':
    images, _ = demo_images(157, 28, 28, classes=4, seed=cfg.seed)
    values = torch.from_numpy(images.transpose(0, 3, 1, 2))
    train_data = TensorDataset(values[:100], torch.zeros(100, dtype=torch.long))
    test_data = TensorDataset(values[100:], torch.zeros(57, dtype=torch.long))
elif cfg.source == 'builtin':
    train_data = datasets.FashionMNIST(cfg.data, train=True, transform=transforms.ToTensor(), download=cfg.allow_download)
    test_data = datasets.FashionMNIST(cfg.data, train=False, transform=transforms.ToTensor(), download=cfg.allow_download)
    if cfg.quick:
        train_data = torch.utils.data.Subset(train_data, range(128))
        test_data = torch.utils.data.Subset(test_data, range(57))
else:
    raise ValueError('Выберите demo или builtin')
batch_size, latent_dim, hidden = 32, 80, 400
loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0, generator=torch.Generator().manual_seed(cfg.seed))
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=0)
print('train/test:', len(train_data), len(test_data))


## VAE и loss
Используются обычные tensors, без устаревшего Variable.


In [ ]:
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden)
        self.mu, self.logvar = nn.Linear(hidden, latent_dim), nn.Linear(hidden, latent_dim)
        self.fc3, self.fc4 = nn.Linear(latent_dim, hidden), nn.Linear(hidden, 784)
    def encode(self, x):
        h = F.relu(self.fc1(x.reshape(-1, 784)))
        return self.mu(h), self.logvar(h)
    def decode(self, z):
        return torch.sigmoid(self.fc4(F.relu(self.fc3(z))))
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(.5 * logvar)
        return self.decode(z), mu, logvar
def loss_function(reconstruction, x, mu, logvar):
    bce = F.binary_cross_entropy(reconstruction, x.reshape(-1, 784), reduction='sum')
    kl = -.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kl
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=.001)
for epoch in range(1 if cfg.quick else 15):
    model.train(); total = 0.
    for batch, _ in loader:
        batch = batch.to(device); optimizer.zero_grad()
        reconstruction, mu, logvar = model(batch)
        loss = loss_function(reconstruction, batch, mu, logvar)
        loss.backward(); optimizer.step(); total += loss.item()
    print('epoch', epoch + 1, 'train loss/sample', total / len(train_data))


## Независимый test и генерация
Форма определяется фактическим batch, а не глобальным batch_size.


In [ ]:
model.eval(); total = 0.
with torch.inference_mode():
    for batch, _ in test_loader:
        batch = batch.to(device)
        reconstruction, mu, logvar = model(batch)
        total += loss_function(reconstruction, batch, mu, logvar).item()
        assert reconstruction.shape[0] == batch.shape[0]
    sample = model.decode(torch.randn(16, latent_dim, device=device)).reshape(-1, 1, 28, 28).cpu()
print('test loss/sample:', total / len(test_data))
destination = cfg.output / 'sample.png'
save_image(sample, destination)
with Image.open(destination) as image:
    plt.imshow(image); plt.axis('off'); plt.show()


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Сравните latent_dim 2/20/80.
2. Объясните роль KL.
3. Измените коэффициент KL (β-VAE).
4. Проверьте последний неполный batch.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
